# Handling Imbalance

**Notebook 5 of 5.** Notebooks 01 to 04 measured the imbalance honestly. This one does
something about it, in four ways:

| Strategy | Idea |
| :-- | :-- |
| **1. Cost-sensitive learning** | during training, count every mistake on the rare class more heavily |
| **2. Resampling** | change the training data: fewer majority rows, or more minority rows |
| **3. Moving the threshold** | keep the model, change the cut-off, the dial from notebook 04 |
| **4. Models built for imbalance** | models with strategy 1 or 2 built into their own training |

The first three are tried below, and compared by **precision, recall and F2**. F2, because
for the bank a missed fraud is worse than a false alarm. The fourth belongs to models
that come later in the program, so it is only named here.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import precision_score, recall_score, fbeta_score

The same steps as in notebook 01, in one cell: load, split, scale, and train a logistic
regression.

In [ ]:
df = pd.read_csv("../data/transactions.csv")
X = df.drop(columns="fraud")
y = df["fraud"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(max_iter=1000).fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)

Each recipe gets one row in a results table, built by this helper.

In [ ]:
def evaluate(name, pred):
    """Score test predictions with the three metrics of this notebook.

    Parameters
    ----------
    name : str
        A label for the recipe.
    pred : array-like
        Predicted classes for the test set, 0 or 1.

    Returns
    -------
    dict
        The recipe's name, number of alarms, precision, recall and F2.
    """
    return {
        "recipe": name,
        "alarms": int(pred.sum()),
        "precision": precision_score(y_test, pred),
        "recall": recall_score(y_test, pred),
        "F2": fbeta_score(y_test, pred, beta=2),
    }


results = [evaluate("plain model, threshold 0.5", y_pred)]
pd.DataFrame(results).round(3)

Every recipe below is fixed in advance and then scored once on the test set, so the
comparison stays fair.

## 1. Cost-sensitive learning: class weights

First, what training actually does. For every training row, logistic regression measures
how far its predicted probability is from the truth: a fraud given a probability of 0.9 is
a small error, a fraud given 0.01 a large one. Fitting means finding the coefficients that
make the **total error over all training rows** as small as possible.

By default, every row's error counts once. The training set has 37,131 honest payments and
369 frauds, so that total is almost entirely about honest payments, and the fit can
barely tell whether it gets the frauds right.

A **class weight** is a number that multiplies the error of every row in a class before
the total is taken. `class_weight="balanced"` chooses the weights so that **both classes
contribute equally** to the total:

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

weights = compute_class_weight(class_weight="balanced", classes=np.array([0, 1]), y=y_train)
counts = y_train.value_counts().sort_index().to_numpy()   # honest, then fraud

pd.DataFrame({"rows": counts, "weight": weights.round(3), "rows x weight": (counts * weights).round()},
             index=["honest (0)", "fraud (1)"])

Each fraud now counts about **100 times** as much as an honest payment, and the two
classes carry the same weight in total. Missing one fraud costs the model as much as
misjudging a hundred honest payments, so it can no longer afford to ignore frauds.

In [ ]:
model_weighted = LogisticRegression(max_iter=1000, class_weight="balanced")
model_weighted.fit(X_train_scaled, y_train)

results.append(evaluate("class weights", model_weighted.predict(X_test_scaled)))
pd.DataFrame(results).round(3)

Recall jumps to 0.90, and precision collapses: **over 400 alarms**, most of them false.
The weights pushed the model hard toward fraud, harder than the bank wanted, and F2 fell.

## 2. Resampling

Resampling changes the training data instead of the model.

- **Undersampling** drops majority rows at random, until the classes are closer in size.
- **Oversampling** adds minority rows, either by copying them at random, or by making up
  new ones. The best-known way to make them up is **SMOTE**, which places a synthetic row
  on the line between a minority row and one of its minority neighbors:

<img src="assets/smote.png" alt="Left: a few orange minority points among many blue ones. Middle: new green synthetic points placed on lines between neighboring orange points. Right: the minority class now has many more points" width="640">

Synthetic rows are not real payments, and too many of them can teach the model patterns
that do not exist. Use them with care.

**Resample the training set only, after the split.** The test set has to look like the
real world, imbalance included. And oversampling before the split would put copies of the
same fraud on both sides, which is test leakage.

> **Exercise:**
>
> Undersample the training set: keep all its frauds, plus a random **10 honest payments
> for every fraud**. Train a plain logistic regression on the result, add its test scores
> to `results`, and show the table.

<details><summary>
Click here for a hint...
</summary>

Work with row positions. `np.where(y_train == 1)[0]` gives the positions of the frauds,
and `rng.choice(positions, size=n, replace=False)` draws `n` of the honest ones, with
`rng = np.random.default_rng(42)`. `np.concatenate` joins the two, and
`X_train_scaled[rows]` and `y_train.iloc[rows]` select those rows.
</details>

In [ ]:
rng = np.random.default_rng(42)
fraud_rows = np.where(y_train == 1)[0]
honest_rows = np.where(y_train == 0)[0]

keep = rng.choice(honest_rows, size=10 * len(fraud_rows), replace=False)
rows = np.concatenate([fraud_rows, keep])
print("training rows:", len(rows), "instead of", len(y_train))

model_under = LogisticRegression(max_iter=1000)
model_under.fit(X_train_scaled[rows], y_train.iloc[rows])

results.append(evaluate("undersampled, 10 to 1", model_under.predict(X_test_scaled)))
pd.DataFrame(results).round(3)

<details><summary>
Click here to compare your answer...
</summary>

The model trains on about 4,000 rows instead of 37,500, and its recall rises above the
plain model's while precision falls, much less than with class weights. Its F2 is
higher than the plain model's. Throwing away nine in ten honest payments sounds wasteful,
and it is: it works here because the honest payments are so plentiful.
</details>

## 3. Moving the threshold

The simplest recipe leaves the model alone and lowers the cut-off, as in notebook 04.

> **Exercise:**
>
> Take the plain model's probabilities and predict fraud from **0.1** upward. Add the
> result to `results` and show the final table.

In [ ]:
proba_test = model.predict_proba(X_test_scaled)[:, 1]
pred_low = (proba_test >= 0.1).astype(int)

results.append(evaluate("plain model, threshold 0.1", pred_low))
pd.DataFrame(results).round(3)

<details><summary>
Click here to compare your answer...
</summary>

The lower threshold gives the **highest F2 of the four**, with no change to the model or
the data. Precision stays above 0.9 while recall rises to about 0.85.
</details>

## 4. Models built for imbalance

Some models build strategy 1 or 2 into their own training, so you do not apply it by
hand. They belong to model families that come later in the program. When you meet them,
the comparison below applies to them unchanged.

## 5. What the table says

**Every recipe moves the model along the same trade-off**: more frauds caught, more false
alarms. None of them is free, and none is best in general.

- Class weights moved it furthest, too far for this bank.
- Undersampling and the lower threshold moved it a moderate distance, and both beat the
  plain model on F2.
- The lower threshold did it without retraining anything, which is why it is usually the
  first thing to try.

So the question from notebook 02 decides here too. Choose the metric from the cost of the
two mistakes, and then pick the recipe that does best on it.

## Summary

- **Cost-sensitive learning** (`class_weight="balanced"`) multiplies each training
  row's error by a weight for its class, so that the rare class counts as much in total as
  the common one.
- **Resampling** changes the training data: undersampling drops majority rows, oversampling
  copies or synthesizes minority rows (SMOTE). **Resample after the split, training set
  only.**
- **Moving the threshold** changes only the decision, and is often enough.
- **Models built for imbalance**, later in the program, have strategy 1 or 2 built in.
- All of them trade precision for recall. **The metric, chosen from the cost of each
  mistake, decides which trade is right.**

Next, and optional, in `06_mammography_exercise.ipynb`: the whole workflow on a harder
problem, screening mammograms.